In [ ]:
notebookutils.fs.ls("Files/")

In [ ]:
file_path = "Files/club_report_sample_synthetic.csv"
df = spark.read.option("header",True).option("inferSchema",True).csv(file_path)
print("Total rowa: ", df.count())
df.show(5,truncate= False)

In [ ]:
from pyspark.sql import functions as F 
#Count missing value in every column
df.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in df.columns
    ]
).show(truncate=False)

In [ ]:
df.groupBy("ELIGIBLE_FOR_BENEFIT").agg(
    F.count("*").alias("Total_rows"),
    F.sum(
        F.col("ELIGIBLE_STATUS_MESSAGE").isnull().cast("int")
    ).alias("missing_status_message")
).show()

In [ ]:
from pyspark.sql import functions as F

df.groupBy("ELIGIBLE_FOR_BENEFIT").agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.col("ELIGIBLE_STATUS_MESSAGE").isNull().cast("int")
    ).alias("missing_status_messages")
).show()

In [ ]:
duplicate_groups = (
    df.groupBy(
        "SAP_CODE",
        "SCHEME_PERIOD_END_DATE"
    )
    .count()
    .filter(F.col("count")>1)
)

print("Duplicates dealer period groups",duplicate_groups.count())
duplicate_groups.show(10,truncate=False)

In [ ]:
# Make column names safe for Delta tables
for old_name in df.columns:
    new_name = (
        old_name.strip()
        .lower()
        .replace(" ","_")
        .replace("-","_")
        .replace("/","_")
    )
    df = df.withColumnRenamed(old_name,new_name)

#Check the new column name 
print(df.columns)

In [ ]:
(
    df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable("Bronze_club_report") 
)

print("Bronze table record")

In [ ]:
bronze_df = spark.read.table("bronze_club_report")
print("Bronze row count : ", bronze_df.count())

bronze_df.show(5,truncate=False)

In [ ]:
# Read the bronze table 
bronze_df = spark.read.table("bronze_club_report")

#check the column names and data types
bronze_df.printSchema()

In [ ]:
# check the data type of our sales column 
bronze_df.select("scheme_period_sales_avg").printSchema()

# Look at five sales_value
bronze_df.select("scheme_period_sales_avg").show(5)

In [ ]:
from pyspark.sql import functions as F

negative_sales = bronze_df.filter(
    F.col("scheme_period_sales_avg")<0
)

print("Rows with the negative sales :",negative_sales.count())

negative_sales.show(5,truncate=False)

In [ ]:
invalid_dates = bronze_df.filter(
    F.col("scheme_period_start_date").isNull()|
    F.col("scheme_period_end_date").isNull()|
    (
        F.col("scheme_period_end_date")<
        F.col("scheme_period_start_date")
    )
)

print("Rows with invalid Dates : ", invalid_dates.count())

invalid_dates.show(5,truncate=False)

In [ ]:
missing_sap_code = bronze_df.filter(
    F.col("sap_code").isNull()|
    (F.trim(F.col("sap_code"))=="")

)

print("Rows with missing Sap Codes :",missing_sap_code.count())
missing_sap_code.show(5, truncate=False)

In [ ]:
# Silver Table
from pyspark.sql import functions as F 
from pyspark.sql.types import DecimalType, IntegerType

# read bronze table
silver_df = spark.read.table("bronze_club_report")

# Read our bronze table
silver_df = (
     silver_df
    .withColumn("sap_code",F.col("sap_code").cast("string"))
    .withColumn("humbee_code",F.col("humbee_code").cast("string"))
    .withColumn(
        "schemePeriod_sales_avg",
        F.col("scheme_period_sales_avg").cast(DecimalType(18,3))
    )
    .withColumn("sales_club_rank",F.col("sales_club_rank").cast(IntegerType()))
    .withColumn("policy_club_rank", F.col("policy_club_rank").cast(IntegerType()))

)
print("Silver preparation complete!")
print("Rows : ", silver_df.count())

In [ ]:
# save silver table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_club_report")
)

print("silver table created sucessfully")

In [ ]:
silver_check = spark.read.table("silver_club_report")

print("silver rows count :", silver_check.count())

silver_check.show(5,truncate=False)

In [ ]:
from pyspark.sql import functions as F 

invalid_sales = silver_check.filter( 
    F.col("scheme_period_sales_avg")<0
)

print("Rows with negative sales:", invalid_sales.count())
invalid_sales.show(5,truncate=False)

In [ ]:
missing_firm = silver_check.filter(
    F.col("firm_name").isNull()|
    (F.trim(F.col("firm_name"))=="")
)

print("Rows with missing firm name :",missing_firm.count())
missing_firm.show(5,truncate=False)

In [ ]:
# Gold Table 1 - How many firms are in each club?
from pyspark.sql import functions as F

silver_df = spark.table("silver_club_report")

(
    silver_df.groupBy("external_state","external_district","club_name")
    .agg(
        F.countDistinct("sap_code").alias('firm_count'),
        F.round(
            F.sum("scheme_period_sales_avg"),2
        ).alias("total_sales_avg"),
        F.count("*").alias("record_count")
        
    )
    .orderBy("external_state","external_district","club_name")
    .show(20,truncate=False)
)

In [ ]:
# turn the resuabel result into gold table
gold_club_summary = (
    silver_df
    .groupBy("external_state","external_district","club_name")
    .agg(
        F.countDistinct("sap_code").alias("firm_count"),
        F.round(
            F.sum("scheme_period_sales_avg"),2
        ).alias("total_sales_avg"),
        F.count("*").alias("record_count")
    )
)

(
    gold_club_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_club_summary")
)

In [ ]:
# validating save table
spark.table("gold_club_summary").show(20,truncate=False)

In [ ]:
silver_df.printSchema()
silver_df.show(5,truncate=False)


In [ ]:
from pyspark.sql import functions as F 

#Read the existing silver Table
silver_df = spark.table("silver_club_report")

# Build a bussiness-ready benefit status table 
gold_firm_benefit_status = (
    silver_df 
    .select(
        "sap_code",
        "humbee_code",
        "firm_name",
        
    )
)

In [ ]:
#Read existing silver table
silver_df = spark.table("silver_club_report")

#build a bussiness ready benefir status table
gold_firm_benefit_status = (
    silver_df
    .select(
        "sap_code",
        "humbee_code",
        "firm_name",
        "firm_type",
        "external_state",
        "external_district",
        "scheme_period_start_date",
        "scheme_period_end_date",
        "scheme_period_month",
        "club_name",
        "policy_club_name",
        "scheme_period_sales_avg",
        "eligible_for_benefit",
        "eligible_status_message"
    )
    .withColumn(
        "benefit_status",
        F.when(
            F.col("eligible_for_benefit")== True,
            F.lit("Eligible")
        )
        .when(
            F.col("eligible_for_benefit")==False,
            F.lit("Not Eligible")
        )
        .otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "benefit_status_reason",
        F.when(
            F.col("eligible_for_benefit").isNull(),
            F.lit("Eligible not available")
        )
        .when(
            F.col("eligible_for_benefit")==True,
            F.lit("Eligible undeer record rules")
        )
        .when(
            F.col("eligible_status_message").isNotNull(),
            F.col("eligible_status_message")
        )
        .otherwise(F.lit("Reason not provided in sources"))
    )
) 


#Save the gold table
(
    gold_firm_benefit_status.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_firm_benefit_status")
)

print("Gold table 2 created successfully")

In [ ]:
gold_df = spark.table("gold_firm_benefit_status")

#check sample record
gold_df.show(5,truncate=False)

# count eligible, ineligible and unkown records
gold_df.groupBy("benefit_status").count().show()

# Investigating the recorded status message
gold_df.groupBy(
    "eligible_for_benefit",
    "eligible_status_message"
).count().show(truncate=False)

In [ ]:
(
    gold_df
    .filter(F.col("benefit_status")=="Not Eligible")
    .select(
        "sap_code",
        "firm_name",
        "scheme_period_month",
        "club_name",
        "policy_club_name",
        "eligible_status_message",
        "benefit_status_reason"
    )
    .show(5,truncate=False)
)

In [ ]:
# Gold table 3 : gold firm club histroy
silver_df = spark.table("silver_club_report")

gold_firm_club_history = (
    silver_df
    .select(
        "sap_code",
        "humbee_code",
        "firm_name",
        "firm_type",
        "external_state",
        "external_district",
        "scheme_period_start_date",
        "scheme_period_end_date",
        "scheme_period_month",
        "club_name",
        "policy_club_name",
        "policy_club_rank",
        "scheme_period_sales_avg",
        "eligible_for_benefit"

    )
    .dropDuplicates(
        [
            "sap_code",
            "scheme_period_start_date",
            "scheme_period_end_date",
            "club_name"
        ]
    )
)

(gold_firm_club_history.write
.format("delta")
.mode("overwrite")
.saveAsTable("gold_firm_club_history")
)

print("Gold Table 3 created successfully")

In [ ]:
history_df = spark.table("gold_firm_club_history")

gold_repeat_club_summary = (
    history_df
    .groupBy("sap_code","firm_name","club_name")
    .agg(
        F.countDistinct(
            "scheme_period_start_date"
        ).alias("periods_in_club"),
        F.min("scheme_period_start_date").alias("first_period"),
        F.max("scheme_period_end_date").alias("last_period"),
        F.countDistinct("eligible_for_benefit").alias(
            "elgibilty_values"
        )
    )
    .filter(F.col("periods_in_club")>=2)
    .orderBy(
        F.desc("periods_in_club"),
        "firm_name"
    )
)

gold_repeat_club_summary.show(5,truncate=False)